In [1]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

from models.r2u_net import R2U_Net
from models.detection_branch import DetectionBranch
from models.matching import OptimalMatching
from models.nms import NonMaxSuppression


ModuleNotFoundError: No module named 'models.r2u_net'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_polyworld(weights_dir):
    backbone = R2U_Net().to(device).eval()
    head = DetectionBranch().to(device).eval()
    matching = OptimalMatching().to(device).eval()
    nms = NonMaxSuppression().to(device)

    backbone.load_state_dict(torch.load(Path(weights_dir)/"polyworld_backbone", map_location=device))
    head.load_state_dict(torch.load(Path(weights_dir)/"polyworld_seg_head", map_location=device))
    matching.load_state_dict(torch.load(Path(weights_dir)/"polyworld_matching", map_location=device))

    return backbone, head, matching, nms


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406]).reshape(1,1,3)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225]).reshape(1,1,3)

def normalize_image(img_np):
    return (img_np - IMAGENET_MEAN) / IMAGENET_STD

def sliding_windows(img, win=320, stride=320):
    W, H = img.size
    tiles = []
    coords = []
    for y in range(0, H, stride):
        for x in range(0, W, stride):
            crop = img.crop((x, y, x+win, y+win))
            tiles.append(crop)
            coords.append((x, y))
    return tiles, coords


In [ ]:
def run_polyworld_on_tile(tile_img, backbone, head, matching, nms):
    # convert to normalized torch tensor
    tile_np = np.array(tile_img).astype(np.float32) / 255.0
    tile_norm = normalize_image(tile_np)
    tile_t = torch.from_numpy(tile_norm).permute(2,0,1)[None].to(device)

    with torch.no_grad():
        features = backbone(tile_t)
        occ_grid = head(features)             # [1,1,80,80]
        _, graph_points = nms(occ_grid)       # NMS on occupancy grid
        polygons = matching.predict(tile_t, features, graph_points)

    return occ_grid, graph_points, polygons


In [ ]:
def visualize_tile(tile_img, occ_grid, graph_points, polygons):
    occ = torch.sigmoid(occ_grid[0,0]).cpu().numpy()

    fig, ax = plt.subplots(1,3,figsize=(14,4))

    ax[0].set_title("Input Tile")
    ax[0].imshow(tile_img)
    ax[0].axis("off")

    ax[1].set_title("Occupancy Grid")
    ax[1].imshow(occ, cmap="inferno")
    ax[1].axis("off")

    ax[2].set_title("Graph Points + Polygons")
    ax[2].imshow(tile_img)
    
    # plot graph points
    pts = graph_points[0].cpu().numpy()
    gH, gW = occ.shape
    scale_x = tile_img.size[0] / gW
    scale_y = tile_img.size[1] / gH
    ax[2].scatter(pts[:,1]*scale_x, pts[:,0]*scale_y, s=10, c='red')

    # plot polygons
    for poly in polygons:
        coords = np.array(poly).reshape(-1,2)
        ax[2].plot(coords[:,0], coords[:,1], "-y", linewidth=2)

    ax[2].axis("off")
    plt.show()


In [ ]:
def run_polyworld_full_image(image_path, weights_dir):
    backbone, head, matching, nms = load_polyworld(weights_dir)

    img = Image.open(image_path).convert("RGB")
    tiles, coords = sliding_windows(img)

    all_polygons_global = []

    for (tile, (x0, y0)) in zip(tiles, coords):
        occ, gpts, polys = run_polyworld_on_tile(tile, backbone, head, matching, nms)

        # visualize ONE tile for debugging
        visualize_tile(tile, occ, gpts, polys)

        # convert polygons to global image coordinates
        for poly in polys:
            arr = np.array(poly).reshape(-1,2)
            arr[:,0] += x0
            arr[:,1] += y0
            all_polygons_global.append(arr)

    return all_polygons_global


In [ ]:
image_path = "midwest-flooding_00000014_pre_disaster.png"
weights_dir = "/media/gisense/xihan/250812_tamu_cybertraining_team4/PolyWorld/trained_weights"

polys = run_polyworld_full_image(image_path, weights_dir)
print("Total polygons:", len(polys))
